In [216]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from imblearn.pipeline import Pipeline

In [217]:
# importing data
df = pd.read_csv('../datasets/ds_challenge_v2_1_data.csv')
print(f'Shape: {df.shape}')
df.head()

Shape: (54681, 11)


,id,city_name,signup_os,signup_channel,signup_date,bgc_date,vehicle_added_date,vehicle_make,vehicle_model,vehicle_year,first_completed_date
0,1,Strark,ios web,Paid,1/2/16,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Strark,windows,Paid,1/21/16,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Wrouver,windows,Organic,1/11/16,1/11/16,NaN,NaN,NaN,NaN,NaN
3,4,Berton,android web,Referral,1/29/16,2/3/16,2/3/16,Toyota,Corolla,2016.0,2/3/16
4,5,Strark,android web,Referral,1/10/16,1/25/16,1/26/16,Hyundai,Sonata,2016.0,NaN


In [218]:
# adjust datatypes
df['vehicle_year'] = df['vehicle_year'].astype('float64')
date_cols = ['signup_date','first_completed_date','bgc_date', 'vehicle_added_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], format = '%m/%d/%y')

df['is_driver'] = ~ pd.isna(df['first_completed_date'])

In [252]:
# time between metrics
df['signup_to_bgc'] = (df['bgc_date'] - df['signup_date']) / datetime.timedelta(days = 1)
df['signup_to_vehicle_added'] = (df['vehicle_added_date'] - df['signup_date'])/datetime.timedelta(days = 1)
#df['signup_to_first_completed'] = (df['first_completed_date'] - df['signup_date'])/datetime.timedelta(days = 1)
df['bgc_to_vehicle_added'] = (df['vehicle_added_date'] - df['bgc_date'])/datetime.timedelta(days = 1)
df['bgc_to_first_completed'] = (df['first_completed_date'] - df['bgc_date'])/datetime.timedelta(days = 1)
#df['vehicle_added_to_first_completed'] = (df['first_completed_date'] - df['vehicle_added_date'])/datetime.timedelta(days = 1)


# snapshot date
snap_date = pd.to_datetime('4/1/16',format = '%m/%d/%y')
# days since difference metrics
df['days_since_signup'] = (snap_date - df['signup_date'])/datetime.timedelta(days = 1)
df['days_since_bgc'] = (snap_date - df['bgc_date'])/datetime.timedelta(days = 1)
df['days_since_vehicle_added'] = (snap_date - df['vehicle_added_date'])/datetime.timedelta(days = 1)
df['days_since_last_activity'] = df[['days_since_signup', 'days_since_bgc', 'days_since_vehicle_added']].min(axis = 1)

df['valid_vehicle_yr'] = df['vehicle_year'] >= 2000 # indicator if vehicle is 16 years old or newer
df['vehicle_year'] = df['vehicle_year'].fillna(0) # fill nulls as 0 for vehicle year

df['vehicle_make'] = df['vehicle_make'].fillna('no_vehicle')
df['vehicle_model'] = df['vehicle_model'].fillna('no_vehicle')
df['signup_os'] = df['signup_os'].fillna('no_os')

In [253]:
df['no_bgc'] = df['bgc_date'].isnull().astype(int)
df['no_vehicle'] = df['vehicle_added_date'].isnull().astype(int)

In [254]:
X = df[['city_name', 'signup_os', 'signup_channel', 'vehicle_year', 'signup_to_bgc', 'signup_to_vehicle_added', 'bgc_to_vehicle_added', 
        'days_since_signup', 'days_since_bgc', 'days_since_vehicle_added', 'days_since_last_activity', 'valid_vehicle_yr', 'no_bgc', 'no_vehicle']]
y = df['is_driver'] # label

In [255]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state = 15) # split into train and test data

In [256]:
ohe_cols = ['city_name', 'signup_os', 'signup_channel'] # cols to be one hot encoded
# need to add indicators for nulls vehicle, bgc, days since
ohe = OneHotEncoder(sparse_output = False)
zero_imputer = SimpleImputer(strategy = 'constant', fill_value = 0) # for imputing 0
mean_imputer = SimpleImputer(strategy = 'mean')

In [257]:
# one hot encoding training data
X_train_ohe = ohe.fit_transform(X_train[ohe_cols]) # one hot encoded cols
col_names = ohe.get_feature_names_out()
X_train_ohe = pd.DataFrame(X_train_ohe, columns=col_names)
X_train_no_ohe = X_train.drop(ohe_cols, axis = 1).reset_index().drop('index', axis = 1)

X_train = pd.concat([X_train_no_ohe, X_train_ohe], axis = 1)

In [258]:
# one hot encoding test data
X_test_ohe = ohe.transform(X_test[ohe_cols])
X_test_ohe = pd.DataFrame(X_test_ohe, columns = col_names)
X_test_no_ohe = X_test.drop(ohe_cols, axis = 1).reset_index().drop('index', axis = 1)

X_test = pd.concat([X_test_no_ohe, X_test_ohe], axis = 1)

In [263]:
X_train_impute = mean_imputer.fit_transform(X_train)
X_test_impute = mean_imputer.transform(X_test)

impute_col_names = mean_imputer.get_feature_names_out()
X_train_impute = pd.DataFrame(X_train_impute, columns=impute_col_names)

X_test_impute = pd.DataFrame(X_test_impute, columns=impute_col_names)

In [264]:
lr = LogisticRegression(max_iter = 10000)
lr.fit(X_train_impute, y_train)


LogisticRegression(max_iter=10000)

In [265]:
y_pred_prob = lr.predict_proba(X_test_impute)
y_pred = lr.predict(X_test_impute)

In [266]:
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, precision_score, recall_score

In [ ]:
print(f'Accuracy: {accuracy_score(y_test, y_pred)}')
print(f'ROC AUC: {roc_auc_score(y_test, y_pred_prob[:,1])}')
print(classification_report(y_test, y_pred))
print(f'Precision: {precision_score(y_test, y_pred)}')
print(f'Recall: {recall_score(y_test, y_pred)}')

Accuracy: 0.9451386772325511
ROC AUC: 0.9699280129016468
              precision    recall  f1-score   support

       False       0.97      0.97      0.97     14593
        True       0.75      0.76      0.75      1812

    accuracy                           0.95     16405
   macro avg       0.86      0.87      0.86     16405
weighted avg       0.95      0.95      0.95     16405



In [268]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

In [269]:
dtc = DecisionTreeClassifier()
dtc.fit(X_train_impute, y_train)

y_pred_prob = dtc.predict_proba(X_test_impute)
y_pred = dtc.predict(X_test_impute)

In [ ]:
print(f'Accuracy: {accuracy_score(y_test, y_pred)}')
print(f'ROC AUC: {roc_auc_score(y_test, y_pred_prob[:,1])}')
print(classification_report(y_test, y_pred))
print(f'Precision: {precision_score(y_test, y_pred)}')
print(f'Recall: {recall_score(y_test, y_pred)}')

Accuracy: 0.920451081987199
ROC AUC: 0.792533717290745
              precision    recall  f1-score   support

       False       0.95      0.96      0.96     14593
        True       0.64      0.62      0.63      1812

    accuracy                           0.92     16405
   macro avg       0.80      0.79      0.79     16405
weighted avg       0.92      0.92      0.92     16405



In [ ]:
rand_forest = RandomForestClassifier(n_estimators = 100)
rand_forest.fit(X_train_impute, y_train)

y_pred_prob = rand_forest.predict_proba(X_test_impute)
y_pred = rand_forest.predict(X_test_impute)

In [ ]:
print(f'Accuracy: {accuracy_score(y_test, y_pred)}')
print(f'ROC AUC: {roc_auc_score(y_test, y_pred_prob[:,1])}')
print(classification_report(y_test, y_pred))
print(f'Precision: {precision_score(y_test, y_pred)}')
print(f'Recall: {recall_score(y_test, y_pred)}')

Accuracy: 0.9411155135629381
ROC AUC: 0.9583987015456472
              precision    recall  f1-score   support

       False       0.97      0.97      0.97     14593
        True       0.74      0.72      0.73      1812

    accuracy                           0.94     16405
   macro avg       0.85      0.85      0.85     16405
weighted avg       0.94      0.94      0.94     16405



In [282]:
importances = rand_forest.feature_importances_
impute_col_names[np.argsort(importances)[::-1]]

array(['signup_to_vehicle_added', 'days_since_vehicle_added',
       'bgc_to_vehicle_added', 'vehicle_year', 'signup_to_bgc',
       'days_since_bgc', 'days_since_last_activity', 'no_vehicle',
       'days_since_signup', 'valid_vehicle_yr', 'signup_os_ios web',
       'city_name_Strark', 'city_name_Berton', 'signup_os_android web',
       'signup_channel_Referral', 'signup_os_mac', 'signup_os_windows',
       'signup_channel_Paid', 'signup_channel_Organic', 'no_bgc',
       'city_name_Wrouver', 'signup_os_other', 'signup_os_no_os'],
      dtype=object)

In [ ]:
# Gini Impurities
for i in range(len(rand_forest.feature_importances_)):
    print(f'{impute_col_names[i]}: {rand_forest.feature_importances_[i]}')

vehicle_year: 0.11346579279912111
signup_to_bgc: 0.0718403467402838
signup_to_vehicle_added: 0.17280344617572158
bgc_to_vehicle_added: 0.1219550032536867
days_since_signup: 0.04925583796854768
days_since_bgc: 0.06347295321753503
days_since_vehicle_added: 0.16053760043535778
days_since_last_activity: 0.054265969620128605
valid_vehicle_yr: 0.038291415169325084
no_bgc: 0.006415918554667629
no_vehicle: 0.05185130663278861
city_name_Berton: 0.009692493512101519
city_name_Strark: 0.009915045708867422
city_name_Wrouver: 0.006016076160558832
signup_os_android web: 0.009641124721440126
signup_os_ios web: 0.010874270123816981
signup_os_mac: 0.007859257997071099
signup_os_no_os: 0.005546990611552068
signup_os_other: 0.005866696945454498
signup_os_windows: 0.007327093683557561
signup_channel_Organic: 0.006481518134183326
signup_channel_Paid: 0.007078588294041136
signup_channel_Referral: 0.009545253540191772
